In [ ]:
import pandas as pd
from tqdm import tqdm
import os
from typing import List, Dict
import seaborn as sns
sns.set(style="whitegrid")

In [ ]:
# Same ordering as paper
task_2_name: Dict[str, str] = {
    # Operational outcomes
    'guo_los': 'Long LOS',
    'guo_readmission': '30-Day Readmission',
    'guo_icu': 'ICU Admission',
    # Anticipating lab test results
    'lab_thrombocytopenia': 'Thrombocytopenia',
    'lab_hyperkalemia': 'Hyperkalemia',
    'lab_hypoglycemia': 'Hypoglycemia',
    'lab_hyponatremia': 'Hyponatremia',
    'lab_anemia': 'Anemia',
    # Assignment of new diagnoses
    'new_hypertension': 'Hypertension',
    'new_hyperlipidemia': 'Hyperlipidemia',
    'new_pancan': 'Pancreatic Cancer',
    'new_celiac': 'Celiac',
    'new_lupus': 'Lupus',
    'new_acutemi' : 'Acute MI',
    # Anticipating chest x-ray findings
    'chexpert' : 'Chest X-Ray',
}

task_2_value_type: Dict[str, str] = {
    'new_pancan': 'boolean',
    'new_celiac': 'boolean',
    'new_lupus': 'boolean',
    'new_acutemi' : 'boolean',
    'new_hypertension': 'boolean',
    'new_hyperlipidemia': 'boolean',
    'guo_los': 'boolean',
    'guo_readmission': 'boolean',
    'guo_icu': 'boolean',
    'lab_thrombocytopenia': 'multiclass',
    'lab_hyperkalemia': 'multiclass',
    'lab_hypoglycemia': 'multiclass',
    'lab_hyponatremia': 'multiclass',
    'lab_anemia': 'multiclass',
    'chexpert' : 'multilabel',
}

In [ ]:
path_to_data_csv = '../EHRSHOT_ASSETS/data/ehrshot.csv'
path_to_labels_dir = '../EHRSHOT_ASSETS/benchmark/'
path_to_splits_csv = '../EHRSHOT_ASSETS/splits/person_id_map.csv'

# Overall Stats

In [ ]:
df_dataset = pd.read_csv(path_to_data_csv)
df_split = pd.read_csv(path_to_splits_csv)

In [ ]:
df_dataset[['patient_id', 'visit_id']].drop_duplicates().groupby('patient_id').size().median()

In [ ]:
print("# of events:", df_dataset.shape[0])
print("# of patients:", df_dataset['patient_id'].nunique())
print("# of visits:", df_dataset['visit_id'].nunique())
print("# of train patients", df_split[df_split['split'] == 'train']['omop_person_id'].nunique())
print("# of val patients", df_split[df_split['split'] == 'val']['omop_person_id'].nunique())
print("# of test patients", df_split[df_split['split'] == 'test']['omop_person_id'].nunique())

# Visit Counts

In [ ]:
visit_counts = []
for pid in tqdm(df_dataset['patient_id'].unique()):
    df_events = df_dataset[df_dataset['patient_id'] == pid]
    n_visits: int = df_events['visit_id'].dropna().nunique()
    visit_counts.append((pid, n_visits))
df_visit_counts = pd.DataFrame(visit_counts, columns=['patient_id', 'n_visits'])
df_visit_counts

In [ ]:
print("# of visits per patient (median):", df_visit_counts['n_visits'].median())
print("# of visits per patient (mean):", df_visit_counts['n_visits'].mean())

In [ ]:
import matplotlib.pyplot as plt
plt.hist(df_visit_counts['n_visits'], bins=50)
plt.ylabel("# of patients")
plt.yscale('log')
plt.xlabel("# of visits")
plt.title(f"EHRSHOT full dataset (n={df_visit_counts.shape[0]})")

# Event Counts

In [ ]:
event_counts = []
for pid in tqdm(df_dataset['patient_id'].unique()):
    df_events = df_dataset[df_dataset['patient_id'] == pid]
    n_events: int = df_events.shape[0]
    event_counts.append((pid, n_events))
df_event_counts = pd.DataFrame(event_counts, columns=['patient_id', 'n_events'])
df_event_counts

In [ ]:
print("# of events per patient (median):", df_event_counts['n_events'].median())
print("# of events per patient (mean):", df_event_counts['n_events'].mean())

In [ ]:
import matplotlib.pyplot as plt
plt.hist(df_event_counts['n_events'], bins=50)
plt.ylabel("# of patients")
plt.yscale('log')
plt.xlabel("# of events")
plt.title(f"EHRSHOT full dataset (n={df_event_counts.shape[0]})")

# Timeline Lengths

In [ ]:
timeline_lengths = []
for pid in tqdm(df_dataset['patient_id'].unique()):
    df_ = df_dataset[df_dataset['patient_id'] == pid].sort_values('start', ascending=True)
    first_visit_idx = df_['visit_id'].first_valid_index()
    last_visit_idx = df_['visit_id'].last_valid_index()
    if first_visit_idx is not None and last_visit_idx is not None:
        length: int = pd.to_datetime(df_.loc[last_visit_idx, 'start']) - pd.to_datetime(df_.loc[first_visit_idx, 'start'])
        timeline_lengths.append((pid, length))
df_timeline_lengths = pd.DataFrame(timeline_lengths, columns=['patient_id', 'length'])
df_timeline_lengths

In [ ]:
print("Years between first and last visit per patient (median):", df_timeline_lengths['length'].median().days / 365.25)
print("Years between first and last visit per patient (mean):", df_timeline_lengths['length'].mean().days / 365.25)

In [ ]:
import matplotlib.pyplot as plt
plt.hist(df_timeline_lengths['length'].dt.days / 365.25, bins=50)
plt.ylabel("# of patients")
plt.yscale('log')
plt.xlabel("Time between first and last visit (years)")
plt.title(f"EHRSHOT patients with at least one visit (n={df_timeline_lengths.shape[0]})")

# Label Stats

In [ ]:
df_labels = pd.read_csv(os.path.join(path_to_labels_dir, 'all_labels.csv'))

In [ ]:
print("# of labels:", df_labels.shape[0])

In [ ]:
results = {
    'train' : [],
    'test' : [],
    'val' : [],
    'all' : []
}
for task, task_name in tqdm(task_2_name.items()):
    path_to_task_csv: str = f"{path_to_labels_dir}{task}/labeled_patients.csv"
    if not os.path.exists(path_to_task_csv):
        print(f"Skipping {task_name} @ {path_to_task_csv}")
        continue
    df = pd.read_csv(path_to_task_csv)
    value_type = task_2_value_type[task]
    if task_2_value_type[task] == "boolean":
        df['is_positive_label'] = df["value"]
    elif task_2_value_type[task] == "multiclass":
        df['is_positive_label'] = df["value"] > 0
    elif task_2_value_type[task] == "multilabel":
        df['is_positive_label'] = df["value"] != 8192
    else:
        print(f"Skipping {task_name}")
        continue
    
    # Splits
    for split in ['train', 'test', 'val']:
        df_ = df[df['patient_id'].isin(df_split[df_split['split'] == split]['omop_person_id'])]
        results[split].append({
            'task' : task,
            'task_name' : task_name,
            'n_patients' : df_['patient_id'].nunique(),
            'n_positive_patients' : df_.groupby('patient_id')['is_positive_label'].max().sum(),
            'n_labels' : df_.shape[0],
            'n_positive_labels' : df_['is_positive_label'].sum(),
        })
    
    # All
    results['all'].append({
        'task' : task,
        'task_name' : task_name,
        'n_patients' : df['patient_id'].nunique(),
        'n_positive_patients' : df.groupby('patient_id')['is_positive_label'].max().sum(),
        'n_labels' : df.shape[0],
        'n_positive_labels' : df['is_positive_label'].sum(),
    })

for key in results.keys():
    results[key] = pd.DataFrame(results[key])
    results[key]['n_negative_labels'] = results[key]['n_labels'] - results[key]['n_positive_labels']
    results[key]['n_negative_patients'] = results[key]['n_patients'] - results[key]['n_positive_patients']
    results[key]['label_prevalence'] = results[key]['n_positive_labels'] / results[key]['n_labels']
    results[key] = results[key][['task_name', 'n_patients', 'n_positive_patients', 'n_negative_patients', 'n_labels', 'n_positive_labels', 'n_negative_labels', 'label_prevalence']]

In [ ]:
results['all']

In [ ]:
# Train
results['train']

In [ ]:
# Val
results['val']

In [ ]:
# Test
results['test']

In [ ]:
print(results['all'].to_markdown(index=False))

In [ ]:
print(results['train'].to_markdown(index=False))

In [ ]:
print(results['val'].to_markdown(index=False))

In [ ]:
print(results['test'].to_markdown(index=False))